# AI Meeting Minutes Generator

This app records audio (or takes an upload), transcribes it using Google Speech Recognition, and then uses an LLM (via OpenRouter) to generate concise meeting minutes.

### Prerequisites
- `gradio`
- `openai`
- `SpeechRecognition`
- `pydub`
- `ffmpeg` (installed on system)
- `python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import whisper

# Load environment variables
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables.")
else:
    print("OPENROUTER_API_KEY loaded.")

# Initialize OpenAI client for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

print("Loading Whisper model...")
whisper_model = whisper.load_model("base")

In [ ]:
def transcribe_audio(audio_path):
    """
    Transcribes audio file using local Whisper AI model.
    """
    try:
        result = whisper_model.transcribe(audio_path)
        return result["text"]
    except Exception as e:
        return f"Error during transcription: {str(e)}"

def generate_minutes(transcript):
    """
    Uses LLM to generate meeting minutes from the transcript.
    """
    if not transcript or "Error" in transcript:
        return "No valid transcript to process."
        
    prompt = f"""
    You are a professional secretary. 
    You produce minutes of meetings from transcripts, with summary, key discussion points, decisions made,
    takeaways and action items with owners, in markdown format without code blocks.

    Transcript:
    {transcript}
    """
    
    try:
        response = client.chat.completions.create(
            model="google/gemini-2.0-flash-001",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes meetings."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating minutes: {str(e)}"

In [ ]:
def process_meeting(audio_path):
    if not audio_path:
        return "Please record or upload audio first.", ""
    
    transcript = transcribe_audio(audio_path)
    
    minutes = generate_minutes(transcript)
    
    return transcript, minutes

# Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#Voice-to-Meeting Minutes App")
    gr.Markdown("Record your meeting or upload an audio file to get an instant summary.")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Audio Input")
            process_btn = gr.Button("Generate Minutes", variant="primary")
        
        with gr.Column(scale=1):
            transcript_output = gr.Textbox(label="Transcript", lines=10)
            minutes_output = gr.Markdown(label="Meeting Minutes")

    process_btn.click(
        fn=process_meeting,
        inputs=[audio_input],
        outputs=[transcript_output, minutes_output]
    )

if __name__ == "__main__":
    demo.launch()

/var/folders/6m/3v81d3yj54z2fqnfs4v_zxdr0000gn/T/ipykernel_49895/635975583.py:14: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
